# Рубежный контроль №1 — Дом и квартира
## Qwen2.5-3B-Instruct: SFT по публичным данным курса

Используется только тема `home`: 100 QA, 50 SFT train, 20 validation, 30 benchmark. Bench-рубрики не показываются генеративной модели. Выберите GPU T4. Новые GPU-результаты должны быть получены фактическим запуском; ноутбук не содержит заранее подставленных оценок.

In [ ]:
from google.colab import files
import pathlib, zipfile, os
ROOT=pathlib.Path('/content/home_course_submission')
if not ROOT.exists():
    uploaded=files.upload()
    zname=next((n for n in uploaded if n.endswith('.zip')),None)
    if not zname: raise RuntimeError('Загрузите home_course_submission.zip')
    with zipfile.ZipFile(zname) as z: z.extractall('/content')
if not (ROOT/'src').exists():
    raise RuntimeError('Не найдена папка проекта home_course_submission/src')
os.chdir(ROOT)
print('Project:',ROOT)

## 1. Установка окружения и проверка GPU

In [ ]:
%cd /content/home_course_submission
!pip -q install -r requirements.txt
import torch
print('CUDA:',torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

## 2. Проверка данных и тестов

In [ ]:
!python -m src.validate_data
!python -m unittest discover -s tests -v

Проверяется число записей, поля и метки, уникальность id, отсутствие пересечения qa_id между train/val/bench, а также совместимость с `datasets.load_dataset`. Семантическое сходство между вопросами этой проверкой не исключается.

## 3. Baseline на validation и benchmark

In [ ]:
!python -m src.generate --split validation --model base
!python -m src.generate --split bench --model base

## 4. QLoRA SFT

In [ ]:
!python -m src.train

Используются только 50 train-примеров; 20 validation-примеров применяются для выбора лучшего checkpoint. План параметров находится в `configs/experiment.json`. Адаптер и токенизатор сохраняются в `artifacts/course_adapter/`.

## 5. Генерация после SFT и автоматические метрики

In [ ]:
!python -m src.generate --split validation --model sft
!python -m src.generate --split bench --model sft
!python -m src.evaluate

## 6. Экспорт заданий независимому судье

In [ ]:
!python -m src.export_bench_judge
from pathlib import Path
p=Path('artifacts/bench_judge_tasks.jsonl')
print('Judge tasks:',sum(1 for _ in p.open(encoding='utf-8')))

Экспорт содержит 60 заданий (30 baseline + 30 SFT). Передайте их независимой модели-судье пакетами. Судья оценивает каждый критерий рубрики true/false с коротким обоснованием. Верните JSONL с полями `task_id` и `scores`, где у каждого score есть `criterion_index`, `met`, `reason`. Сохраните полный результат как `artifacts/bench_judge_results.jsonl`. Не показывайте модели-генератор rubric и эталонные метки.

In [ ]:
# Выполнить после сохранения всех 60 реальных судейских оценок:
# !python -m src.score_bench_judge

## 7. Отчёт и артефакты

Заполните `reports/REPORT_TEMPLATE.md` только фактическими результатами Colab. Сохраните предсказания, train/eval logs, метрики, графики, benchmark judge results и SHA-256 адаптера. Не подставляйте результаты предыдущего синтетического эксперимента вместо метрик на данных курса.